# Análise de Deslocamentos InSAR + Modelo HST para Barragens

**Fluxo de trabalho:**
1. Leitura e filtragem dos dados InSAR (EGMS, ascendente + descendente)
2. Interpolação temporal e combinação por IDW
3. Decomposição em componentes vertical (dV) e horizontal (dH)
4. Geração de grelha e agregação por célula
5. Ajuste do modelo HST (Hidrostático–Sazonal–Tempo) para uma célula à escolha
6. Validação e visualização dos resultados

## 0. Imports e Configuração Global

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import contextily as ctx
from shapely.geometry import box
from shapely.strtree import STRtree
from scipy.spatial import cKDTree
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ============================================================
# CONFIGURAÇÃO CENTRAL — edite apenas aqui
# ============================================================

# Variável de deslocamento a analisar: 'dV' (vertical) ou 'dH' (horizontal)
TARGET_VAR = 'dV'

# Tamanho das células da grelha (metros, sistema EPSG:3035)
GRID_SIZE = 50

# Célula a usar no modelo HST (ID numérico — ver Figura 1)
CELULA_HST = 69

# Barragem — coordenadas da área de interesse (EPSG:3035)
NORTE_MIN, NORTE_MAX = 1855050, 1855850   # Alqueva
ESTE_MIN,  ESTE_MAX  = 2792250, 2793250

# Ficheiros InSAR
ASC_FILE  = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
DESC_FILE = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# Ficheiros de variáveis externas
NIVEL_FILE = "data/alqueva_nivel.xlsx"   # colunas esperadas: 'data', 'nivel'
TEMP_FILE  = "data/alqueva_temp.xlsx"    # colunas esperadas: 'data', 'med'

# COS para máscara de cobertura do solo (opcional)
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"

print("✅ Configuração carregada.")

## 1. Leitura dos Dados InSAR e da Máscara de Barragem

In [ ]:
# --- 1a. Máscara COS (infraestrutura hídrica) ---
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
    print(f"COS lido: {len(barragem)} polígono(s) de barragem encontrado(s).")
except Exception as e:
    print(f"Aviso COS: {e}\n→ A usar bounding box manual como máscara.")
    barragem = gpd.GeoDataFrame(
        {'geometry': [box(ESTE_MIN, NORTE_MIN, ESTE_MAX, NORTE_MAX)]},
        crs="EPSG:3035"
    )

# --- 1b. CSVs InSAR ---
def filter_area(df):
    return df[
        (df['northing'] >= NORTE_MIN) & (df['northing'] <= NORTE_MAX) &
        (df['easting']  >= ESTE_MIN)  & (df['easting']  <= ESTE_MAX)
    ].copy()

asc  = filter_area(pd.read_csv(ASC_FILE))
desc = filter_area(pd.read_csv(DESC_FILE))
print(f"Pontos após filtro geográfico → Ascendente: {len(asc)}, Descendente: {len(desc)}")

## 2. Pré-processamento InSAR: Melt, Interpolação Temporal e IDW

In [ ]:
def melt_to_long(df):
    """Converte colunas de datas para formato longo (tidy)."""
    disp_cols = df.columns[24:]   # As colunas a partir do índice 24 são datas
    long_df = df.melt(
        id_vars=['easting', 'northing', 'incidence_angle', 'track_angle', 'latitude', 'longitude'],
        value_vars=disp_cols,
        var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp', 'date'])


def interpolate_ps(df, dates):
    """Interpola cada ponto PS para as datas mensais comuns."""
    dfs = []
    for (x, y), g in df.groupby(['easting', 'northing']):
        g = g.sort_values('date')
        interp = np.interp(
            pd.to_datetime(dates).astype(np.int64),
            g['date'].astype(np.int64),
            g['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0],
            'incidence_angle': g['incidence_angle'].iloc[0],
            'track_angle': g['track_angle'].iloc[0],
            'date': dates, 'disp': interp
        }))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def idw_combine(source, target, radius=150, power=2):
    """IDW: interpola os valores de 'source' para as localizações de 'target'."""
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date'] == d].copy()
        if src.empty or tgt.empty:
            continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(
            list(zip(tgt['easting'], tgt['northing'])),
            k=5, distance_upper_bound=radius
        )
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m):
                vals.append(np.nan); thetas.append(np.nan)
                continue
            w = 1 / (d_i[m] ** power)
            vals.append(np.sum(w * src.iloc[i_i[m]]['disp']) / np.sum(w))
            thetas.append(np.sum(w * src.iloc[i_i[m]]['incidence_angle']) / np.sum(w))
        tgt['disp_idw'] = vals
        tgt['theta_desc'] = thetas
        out.append(tgt)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


# Pipeline
asc_long  = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)
print(f"Período comum: {common_dates[0].date()} → {common_dates[-1].date()} ({len(common_dates)} meses)")

asc_interp  = interpolate_ps(asc_long,  common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# IDW: projectar descente sobre posições ascendentes
asc_interp = idw_combine(desc_interp, asc_interp).dropna(subset=['disp_idw'])
print(f"Pontos após IDW: {len(asc_interp)}")

## 3. Decomposição em Componentes dV e dH

In [ ]:
# Ângulo da órbita polar (inclinação típica Sentinel-1)
ORB_INC = np.deg2rad(98.6)

asc_interp['beta'] = np.arcsin(
    np.cos(ORB_INC) * np.cos(np.deg2rad(asc_interp['latitude']))
)

def get_comps(row):
    """Decomposição geométrica LOS → dV e dH."""
    ta   = np.deg2rad(row['incidence_angle'])   # ângulo ascendente
    td   = np.deg2rad(row['theta_desc'])         # ângulo descendente (IDW)
    beta = row['beta']
    denom = (
        np.cos(ta) * np.sin(td) * np.cos(beta)
        + np.cos(td) * np.sin(ta) * np.cos(beta)
    )
    if denom == 0:
        return np.nan, np.nan
    dV = (
        row['disp_idw'] * np.sin(ta) * np.cos(beta)
        + row['disp']    * np.sin(td) * np.cos(beta)
    ) / denom
    dH = (
        row['disp_idw'] * np.cos(ta)
        - row['disp']   * np.cos(td)
    ) / denom
    return dV, dH

asc_interp[['dV', 'dH']] = asc_interp.apply(
    lambda x: pd.Series(get_comps(x)), axis=1
)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])
print(f"Pontos com dV e dH válidos: {len(asc_interp)}")

## 4. Grelha, Agregação e Recorte pela Máscara da Barragem

In [ ]:
# --- Criar grelha regular ---
xe = np.arange(asc_interp['easting'].min(),  asc_interp['easting'].max()  + GRID_SIZE, GRID_SIZE)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max() + GRID_SIZE, GRID_SIZE)
xs, ys = xe - GRID_SIZE / 2, ye - GRID_SIZE / 2

asc_interp['cx'] = pd.cut(asc_interp['easting'],  bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx', 'cy'])
asc_interp['cell_id'] = (
    asc_interp['cx'].astype(int).astype(str) + "_" +
    asc_interp['cy'].astype(int).astype(str)
)

grid_data = [
    {'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])}
    for ix in range(len(xs)-1) for iy in range(len(ys)-1)
]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# --- Agregação por célula e data ---
agg = asc_interp.groupby(['cell_id', 'date']).agg(
    val=(TARGET_VAR, 'mean')
).reset_index().rename(columns={'val': TARGET_VAR})

# --- Recorte pela máscara da barragem ---
points_gdf   = gpd.GeoDataFrame(
    asc_interp,
    geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

grid_3857    = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort  = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values
ids  = grid_recort['cell_id'].values
tree = STRtree(poly)
valid_cells = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt):
            valid_cells.add(ids[idx])

# --- Células finais com IDs simples ---
grid_sel = grid_recort[grid_recort['cell_id'].isin(valid_cells)].copy()
grid_sel = grid_sel.sort_values('cell_id').reset_index(drop=True)
grid_sel['simple_id'] = range(1, len(grid_sel) + 1)

id_map = dict(zip(grid_sel['cell_id'], grid_sel['simple_id']))
agg    = agg[agg['cell_id'].isin(valid_cells)].copy()
agg['simple_id'] = agg['cell_id'].map(id_map)

agg_pivot = agg.pivot(index='simple_id', columns='date', values=TARGET_VAR)

n_cells = len(grid_sel)
print(f"✅ Total de células válidas na área da barragem: {n_cells}")

## 5. Figura 1 — Mapa das Células

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Todas as células da grelha (fundo, transparente)
grid.to_crs(epsg=3857).boundary.plot(ax=ax, color='white', lw=0.3, alpha=0.3)

# Células seleccionadas — destaque da célula HST
for _, row in grid_sel.iterrows():
    is_target = (row['simple_id'] == CELULA_HST)
    color  = 'yellow' if is_target else 'none'
    lw     = 2.5     if is_target else 1.5
    ec     = 'red'   if is_target else 'black'
    gpd.GeoDataFrame([row], crs=grid_sel.crs).plot(
        ax=ax, facecolor=color, edgecolor=ec, linewidth=lw, alpha=0.4 if is_target else 1
    )
    cent = row.geometry.centroid
    ax.text(
        cent.x, cent.y, str(row['simple_id']),
        fontsize=11, color='red' if is_target else 'white',
        fontweight='bold', ha='center', va='center'
    )

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_axis_off()
ax.set_title(
    f"Células InSAR na Barragem ({n_cells} células)\n"
    f"Célula seleccionada para HST: {CELULA_HST} (a amarelo)",
    fontsize=13
)
plt.tight_layout()
plt.show()

## 6. Figura 2 — Séries Temporais de Todas as Células

In [ ]:
cols = int(np.ceil(np.sqrt(n_cells)))
rows = int(np.ceil(n_cells / cols))

fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows), sharex=True, sharey=True)
axes = axes.flatten()

ymin = agg[TARGET_VAR].min()
ymax = agg[TARGET_VAR].max()
pad  = (ymax - ymin) * 0.1

for i, ax in enumerate(axes):
    if i < n_cells:
        sid  = i + 1
        data = agg_pivot.loc[sid]
        color = 'red' if sid == CELULA_HST else 'black'
        lw    = 1.8   if sid == CELULA_HST else 1.0
        ax.plot(data.index, data.values, color=color, lw=lw)
        ax.set_title(f"Célula {sid}", fontsize=9, fontweight='bold',
                     color='red' if sid == CELULA_HST else 'black')
        ax.set_ylim(ymin - pad, ymax + pad)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        if i % cols == 0:
            ax.set_ylabel(f'{TARGET_VAR} (mm)', fontsize=8)
        ax.tick_params(labelsize=7)
    else:
        ax.axis('off')

fig.suptitle(f'Séries Temporais — {TARGET_VAR} por Célula', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Carregamento das Variáveis Externas (Nível e Temperatura)

In [ ]:
# --- Nível da albufeira ---
try:
    df_nivel_raw = pd.read_excel(NIVEL_FILE)
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
    df_nivel = (
        df_nivel_raw.set_index('data')
        .resample('MS').mean()
        .reset_index()
        .rename(columns={'nivel': 'nivel'})
    )
    print(f"Nível carregado: {len(df_nivel)} observações mensais.")
except Exception as e:
    print(f"Aviso (nível): {e} → A usar dados sintéticos.")
    d_rng = common_dates
    df_nivel = pd.DataFrame({
        'data': d_rng,
        'nivel': 140 + 10 * np.sin(np.arange(len(d_rng)) * np.pi / 6)
    })

# --- Temperatura ---
try:
    df_temp_raw = pd.read_excel(TEMP_FILE)
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
    df_temp = (
        df_temp_raw.set_index('data')
        .resample('MS').mean()
        .reset_index()
        .rename(columns={'med': 'med'})
    )
    print(f"Temperatura carregada: {len(df_temp)} observações mensais.")
except Exception as e:
    print(f"Aviso (temperatura): {e} → A usar dados sintéticos.")
    df_temp = pd.DataFrame({
        'data': common_dates,
        'med': 15 + 8 * np.cos(np.arange(len(common_dates)) * np.pi / 6)
    })

## 8. Modelo HST para a Célula Seleccionada

O modelo **HST** (Hydrostatic–Seasonal–Time) é o método de referência para barragens.
Decompõe o deslocamento em três efeitos:

| Componente | Símbolo | Preditores |
|---|---|---|
| **H**idrostático | F₁ | h, h², h³, h⁴ |
| **S**azonal (temperatura) | F₂ | cos(2πt/365), sin(2πt/365), cos(4πt/365), sin(4πt/365) |
| **T**empo (irreversível) | F₃ | t (dias), ln(t+1) |

In [ ]:
if CELULA_HST not in agg_pivot.index:
    raise ValueError(f"Célula {CELULA_HST} não encontrada. Verifique o mapa acima e ajuste CELULA_HST.")

# --- 8a. Série InSAR da célula ---
y_series = agg_pivot.loc[CELULA_HST].dropna()
dates    = y_series.index
t_days   = (dates - dates.min()).days.values.astype(float)

# --- 8b. Alinhamento das variáveis externas ---
h    = df_nivel.set_index('data')['nivel'].reindex(dates).interpolate(method='time').values
temp = df_temp.set_index('data')['med'].reindex(dates).interpolate(method='time').values

# Verificar NaN após alinhamento
valid = ~(np.isnan(h) | np.isnan(temp) | np.isnan(y_series.values))
if valid.sum() < 12:
    raise ValueError("Dados insuficientes após alinhamento. Verifique os ficheiros de nível e temperatura.")

dates_v  = dates[valid]
t_v      = t_days[valid]
y_v      = y_series.values[valid]
h_v      = h[valid]
temp_v   = temp[valid]

# --- 8c. Construção da matriz de preditores X ---
s  = 2 * np.pi * t_v / 365.25
s2 = 4 * np.pi * t_v / 365.25

# F1: Hidrostático (polinómio de grau 4 em h)
F1 = np.column_stack([h_v, h_v**2, h_v**3, h_v**4])

# F2: Sazonal — dois harmónicos (anual + semestral)
F2 = np.column_stack([np.cos(s), np.sin(s), np.cos(s2), np.sin(s2)])

# F3: Tempo — linear + logarítmico (fluência)
F3 = np.column_stack([t_v, np.log(t_v + 1)])

X  = np.hstack([F1, F2, F3])

feat_names = [
    'H (h)',  'H (h²)', 'H (h³)', 'H (h⁴)',
    'S cos(ω)', 'S sin(ω)', 'S cos(2ω)', 'S sin(2ω)',
    'T linear', 'T ln(t+1)'
]

print(f"Células: {CELULA_HST} | Observações válidas: {valid.sum()} | Preditores: {X.shape[1]}")

## 9. Figura 3 — Variáveis Externas e Deslocamento InSAR

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(13, 6), sharex=True)
fig.suptitle(
    f'Célula {CELULA_HST} — Deslocamento InSAR e Variáveis Externas',
    fontsize=13, fontweight='bold'
)

# InSAR
axs[0].plot(dates_v, y_v, 'k-o', markersize=3, lw=1.2, label=f"{TARGET_VAR} (InSAR)")
axs[0].set_ylabel(f"{TARGET_VAR} (mm)", fontsize=9)
axs[0].legend(fontsize=8, frameon=False)

# Nível da albufeira
ax_h = axs[1]
ax_h.plot(df_nivel['data'], df_nivel['nivel'], color='steelblue', lw=1.5)
ax_h.fill_between(df_nivel['data'], df_nivel['nivel'], df_nivel['nivel'].min(),
                  alpha=0.15, color='steelblue')
ax_h.set_ylabel('Nível albufeira (m)', fontsize=9)
ax_h.set_xlim(dates_v.min(), dates_v.max())

# Temperatura
axs[2].plot(df_temp['data'], df_temp['med'], color='firebrick', lw=1.5)
axs[2].fill_between(df_temp['data'], df_temp['med'], df_temp['med'].min(),
                    alpha=0.15, color='firebrick')
axs[2].set_ylabel('Temperatura (°C)', fontsize=9)
axs[2].set_xlim(dates_v.min(), dates_v.max())
axs[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axs[2].xaxis.get_majorticklabels(), rotation=30, ha='right')

for ax in axs:
    ax.grid(True, alpha=0.2)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

## 10. Figura 4 — Ajuste Global HST vs InSAR + Contribuições por Componente

In [ ]:
def calc_metrics(real, pred):
    r2   = r2_score(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    mae  = mean_absolute_error(real, pred)
    return r2, rmse, mae

# --- Ajuste global ---
model_full  = LinearRegression().fit(X, y_v)
y_pred_full = model_full.predict(X)
r2_g, rmse_g, mae_g = calc_metrics(y_v, y_pred_full)

# --- Contribuições individuais de cada componente ---
coef = model_full.coef_
intercept = model_full.intercept_

contrib_H = X[:, 0:4]  @ coef[0:4]          # hidrostático
contrib_S = X[:, 4:8]  @ coef[4:8]          # sazonal
contrib_T = X[:, 8:10] @ coef[8:10]         # tempo

# --- Plot ---
fig, axs = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                        gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle(
    f'Célula {CELULA_HST} — Modelo HST: Ajuste Global\n'
    f'R² = {r2_g:.3f}  |  RMSE = {rmse_g:.3f} mm  |  MAE = {mae_g:.3f} mm',
    fontsize=12, fontweight='bold'
)

# Painel superior: série + ajuste + componentes
axs[0].plot(dates_v, y_v,          'k.', alpha=0.5, ms=4, label='InSAR observado')
axs[0].plot(dates_v, y_pred_full,  color='darkgreen', lw=2,   label='HST ajustado')
axs[0].plot(dates_v, contrib_H + intercept, '--',  color='steelblue', lw=1.2, alpha=0.8, label='Componente H (hidrostático)')
axs[0].plot(dates_v, contrib_S,    ':',   color='orange',    lw=1.2, alpha=0.8, label='Componente S (sazonal)')
axs[0].plot(dates_v, contrib_T,    '-.',  color='purple',    lw=1.2, alpha=0.8, label='Componente T (tempo)')
axs[0].set_ylabel(f'{TARGET_VAR} (mm)', fontsize=9)
axs[0].legend(fontsize=8, frameon=False, ncol=3)
axs[0].grid(True, alpha=0.2)

# Painel inferior: resíduos
residuals = y_v - y_pred_full
axs[1].bar(dates_v, residuals, width=20, color=np.where(residuals >= 0, 'steelblue', 'tomato'), alpha=0.7)
axs[1].axhline(0, color='black', lw=0.8)
axs[1].set_ylabel('Resíduo (mm)', fontsize=9)
axs[1].set_xlabel('')
axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axs[1].xaxis.get_majorticklabels(), rotation=30, ha='right')
axs[1].grid(True, alpha=0.2)

for ax in axs:
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

print(f"\n{'='*55}")
print(f" AJUSTE GLOBAL — Célula {CELULA_HST}")
print(f"{'='*55}")
print(f"  R²   : {r2_g:.4f}")
print(f"  RMSE : {rmse_g:.4f} mm")
print(f"  MAE  : {mae_g:.4f} mm")
print(f"{'='*55}")

## 11. Figura 5 — Validação Treino / Teste (sem shuffle temporal)

In [ ]:
# Split temporal 70% treino / 30% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y_v, test_size=0.3, shuffle=False
)
dates_train = dates_v[:len(y_train)]
dates_test  = dates_v[len(y_train):]

model_val = LinearRegression().fit(X_train, y_train)
y_tr_pred = model_val.predict(X_train)
y_te_pred = model_val.predict(X_test)

r2_tr, rmse_tr, mae_tr = calc_metrics(y_train, y_tr_pred)
r2_te, rmse_te, mae_te = calc_metrics(y_test,  y_te_pred)

fig, axs = plt.subplots(2, 1, figsize=(13, 6), sharex=True,
                        gridspec_kw={'height_ratios': [2.5, 1]})
fig.suptitle(
    f'Célula {CELULA_HST} — Validação Treino/Teste (split temporal)',
    fontsize=12, fontweight='bold'
)

# Painel superior
axs[0].plot(dates_v, y_v, color='black', lw=1, alpha=0.25, label='InSAR observado')
axs[0].plot(dates_train, y_tr_pred, 'b--', lw=1.5, label=f'Treino  R²={r2_tr:.3f} | RMSE={rmse_tr:.2f}mm')
axs[0].plot(dates_test,  y_te_pred, 'r-',  lw=2.0, label=f'Teste   R²={r2_te:.3f} | RMSE={rmse_te:.2f}mm')
axs[0].axvline(dates_test[0], color='grey', ls='--', lw=1, alpha=0.7, label='Início do teste')
axs[0].set_ylabel(f'{TARGET_VAR} (mm)', fontsize=9)
axs[0].legend(fontsize=8, frameon=False)
axs[0].grid(True, alpha=0.2)

# Painel inferior: resíduos treino + teste
res_tr = y_train - y_tr_pred
res_te = y_test  - y_te_pred
axs[1].bar(dates_train, res_tr, width=20, color='steelblue', alpha=0.5, label='Resíduo treino')
axs[1].bar(dates_test,  res_te, width=20, color='tomato',    alpha=0.5, label='Resíduo teste')
axs[1].axhline(0, color='black', lw=0.8)
axs[1].axvline(dates_test[0], color='grey', ls='--', lw=1, alpha=0.7)
axs[1].set_ylabel('Resíduo (mm)', fontsize=9)
axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axs[1].xaxis.get_majorticklabels(), rotation=30, ha='right')
axs[1].legend(fontsize=8, frameon=False)
axs[1].grid(True, alpha=0.2)

for ax in axs:
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

print(f"\n{'='*55}")
print(f" VALIDAÇÃO — Célula {CELULA_HST}")
print(f"{'='*55}")
print(f"  TREINO → R²: {r2_tr:.4f}  |  RMSE: {rmse_tr:.4f} mm")
print(f"  TESTE  → R²: {r2_te:.4f}  |  RMSE: {rmse_te:.4f} mm")
overfit = r2_tr - r2_te
print(f"  Diferença R² (overfitting): {overfit:.4f}")
if overfit > 0.15:
    print("  ⚠️  Possível overfitting — considere reduzir o grau de H.")
else:
    print("  ✅  Estabilidade do modelo aceitável.")
print(f"{'='*55}")

## 12. Figura 6 — Importância dos Coeficientes HST

In [ ]:
# Normalizar coeficientes pelo desvio-padrão de cada preditor (importância relativa)
X_std = X.std(axis=0)
coef_norm = np.abs(model_full.coef_ * X_std)

# Agrupar por componente
groups = {
    'Hidrostático (H)': coef_norm[0:4].sum(),
    'Sazonal (S)':      coef_norm[4:8].sum(),
    'Tempo (T)':        coef_norm[8:10].sum(),
}
group_colors = ['steelblue', 'orange', 'purple']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(
    f'Célula {CELULA_HST} — Importância dos Preditores HST',
    fontsize=12, fontweight='bold'
)

# Coeficientes individuais
bar_colors = (['steelblue']*4 + ['orange']*4 + ['purple']*2)
bars = ax1.barh(feat_names, coef_norm, color=bar_colors, edgecolor='white')
ax1.set_xlabel('|coeficiente × σ(preditor)|', fontsize=9)
ax1.set_title('Preditores individuais', fontsize=10)
ax1.tick_params(labelsize=8)
ax1.grid(True, axis='x', alpha=0.3)

# Pizza por componente
wedges, texts, autotexts = ax2.pie(
    list(groups.values()),
    labels=list(groups.keys()),
    colors=group_colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.7
)
for t in autotexts:
    t.set_fontsize(9)
ax2.set_title('Contribuição por componente', fontsize=10)

plt.tight_layout()
plt.show()

## 13. Figura 7 — Decomposição STL das Células (escala uniforme)

In [ ]:
decomps = {}
vals_obs, vals_trend, vals_seas, vals_resid = [], [], [], []

for i in range(n_cells):
    sid    = i + 1
    series = agg_pivot.loc[sid]
    try:
        res = seasonal_decompose(
            pd.Series(series.values, index=series.index),
            period=12, model='additive', extrapolate_trend='freq'
        )
        decomps[sid] = res
        vals_obs.append(res.observed); vals_trend.append(res.trend)
        vals_seas.append(res.seasonal); vals_resid.append(res.resid)
    except:
        decomps[sid] = None

def global_lim(lst):
    if not lst: return (0, 1)
    all_v = pd.concat(lst)
    vmin, vmax = all_v.min(), all_v.max()
    m = (vmax - vmin) * 0.1 or 0.1
    return (vmin - m, vmax + m)

ylim_obs   = global_lim(vals_obs)
ylim_trend = global_lim(vals_trend)
ylim_seas  = global_lim(vals_seas)
ylim_resid = global_lim(vals_resid)

fig, axes = plt.subplots(n_cells, 4, figsize=(16, max(4, 1.5*n_cells)), sharex=True)
if n_cells == 1:
    axes = axes.reshape(1, 4)

for i in range(n_cells):
    sid = i + 1
    res = decomps.get(sid)
    if res is None:
        continue
    lbl_color = 'red' if sid == CELULA_HST else 'black'

    axes[i, 0].plot(res.observed.index, res.observed,  color='black',     lw=1.0)
    axes[i, 1].plot(res.trend.index,    res.trend,     color='steelblue', lw=1.0)
    axes[i, 2].plot(res.seasonal.index, res.seasonal,  color='darkorange', lw=1.0)
    axes[i, 3].scatter(res.resid.index, res.resid,     color='grey', s=5, alpha=0.7)
    axes[i, 3].axhline(0, c='k', ls='--', lw=0.5)

    for j, (lim, col) in enumerate(zip(
        [ylim_obs, ylim_trend, ylim_seas, ylim_resid], range(4)
    )):
        axes[i, j].set_ylim(lim)
        if i == CELULA_HST - 1:
            axes[i, j].set_facecolor('#fff8f0')  # fundo suave na célula HST

    axes[i, 0].set_ylabel(f'Célula {sid}', fontweight='bold', color=lbl_color, fontsize=8)

    if i == 0:
        axes[i, 0].set_title('Observado')
        axes[i, 1].set_title('Tendência')
        axes[i, 2].set_title('Sazonalidade')
        axes[i, 3].set_title('Resíduo')

    if i == n_cells - 1:
        for ax in axes[i, :]:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.suptitle(
    f'Decomposição STL — {TARGET_VAR} (célula HST em destaque: {CELULA_HST})',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

## 14. Resumo Final

In [ ]:
print("\n" + "="*65)
print(f"  RESUMO — MODELO HST | Barragem Alqueva | Célula {CELULA_HST}")
print("="*65)
print(f"  Variável analisada  : {TARGET_VAR}")
print(f"  Período             : {dates_v.min().date()} → {dates_v.max().date()}")
print(f"  Observações         : {len(y_v)}")
print(f"  Preditores HST      : {X.shape[1]}")
print()
print(f"  ── Ajuste Global ──────────────────────────────")
print(f"  R²        : {r2_g:.4f}")
print(f"  RMSE      : {rmse_g:.4f} mm")
print(f"  MAE       : {mae_g:.4f} mm")
print()
print(f"  ── Validação Treino/Teste (70/30) ─────────────")
print(f"  Treino R² : {r2_tr:.4f}  |  RMSE: {rmse_tr:.4f} mm")
print(f"  Teste  R² : {r2_te:.4f}  |  RMSE: {rmse_te:.4f} mm")
print()
print(f"  ── Coeficientes do Modelo ─────────────────────")
print(f"  Intercepto: {model_full.intercept_:.4f} mm")
for name, c in zip(feat_names, model_full.coef_):
    print(f"    {name:<22}: {c:+.5f}")
print("="*65)